# Notebook 0 — Setup & Data Preparation (PCA=300)



In [1]:
import os, warnings, pickle, copy, json
import numpy as np
import pandas as pd
from pathlib import Path

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, roc_auc_score)

warnings.filterwarnings("ignore")
np.random.seed(42)

DATASET_ROOT = "/home/ali/Desktop/Algerian_Lamiaceae_v3"
OUTPUT_DIR   = os.path.join(DATASET_ROOT, "results")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output: {OUTPUT_DIR}")


Device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU
Output: /home/ali/Desktop/Algerian_Lamiaceae_v3/results


## Step 1 — Handcrafted Feature Extraction

In [2]:
import cv2
from PIL import Image
import scipy.stats as stats
from skimage.feature import local_binary_pattern, hog
from skimage.filters import sobel
from skimage.morphology import skeletonize, binary_closing, disk
from tqdm.notebook import tqdm

IMG_SIZE = (224, 224)

def extract_texture(gray):
    lbp1 = local_binary_pattern(gray, P=8,  R=1, method="uniform")
    h1, _ = np.histogram(lbp1.ravel(), bins=10, range=(0,10), density=True)
    lbp2 = local_binary_pattern(gray, P=16, R=2, method="uniform")
    h2, _ = np.histogram(lbp2.ravel(), bins=18, range=(0,18), density=True)
    hf = hog(gray, orientations=9, pixels_per_cell=(16,16),
              cells_per_block=(2,2), feature_vector=True)
    return np.concatenate([h1, h2, [hf.mean(), hf.std(), np.percentile(hf,75)]])

def extract_color(img_rgb):
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    feats = []
    for ch_img in [hsv, lab]:
        for c in range(3):
            ch = ch_img[:,:,c].ravel()
            feats += [ch.mean(), ch.std(), stats.skew(ch), stats.kurtosis(ch)]
    return np.array(feats)

def extract_shape(gray):
    _, binary = cv2.threshold((gray*255).astype(np.uint8), 0, 255,
                               cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    m  = cv2.moments(binary)
    hu = cv2.HuMoments(m).flatten()
    hu = -np.sign(hu) * np.log10(np.abs(hu)+1e-10)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL,
                                    cv2.CHAIN_APPROX_SIMPLE)
    if contours:
        cnt  = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)
        hull = cv2.contourArea(cv2.convexHull(cnt))
        per  = cv2.arcLength(cnt, True)
        x,y,w,h = cv2.boundingRect(cnt)
        extra = np.array([w/(h+1e-6), area/(hull+1e-6), area/(w*h+1e-6),
                           4*np.pi*area/(per**2+1e-6),
                           area/(gray.shape[0]*gray.shape[1])])
    else:
        extra = np.zeros(5)
    return np.concatenate([hu, extra])

def extract_vein(gray):
    sob   = sobel(gray)
    feats = np.array([sob.mean(), sob.std(),
                      np.percentile(sob,25), np.percentile(sob,75),
                      np.percentile(sob,90)])
    _, binary = cv2.threshold((gray*255).astype(np.uint8), 0, 255,
                               cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    bw      = binary_closing(binary.astype(bool), disk(3))
    density = skeletonize(bw).sum()/(gray.shape[0]*gray.shape[1]+1e-6)
    return np.concatenate([feats, [density]])

def extract_handcrafted(img_path):
    img  = Image.open(img_path).convert("RGB").resize(IMG_SIZE)
    arr  = np.array(img)
    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY).astype(np.float32)/255.0
    return np.concatenate([extract_texture(gray), extract_color(arr),
                            extract_shape(gray),  extract_vein(gray)])

HC_COLS = ([f"texture_{i:02d}" for i in range(31)] +
           [f"color_{i:02d}"   for i in range(24)] +
           [f"shape_{i:02d}"   for i in range(12)] +
           [f"vein_{i:02d}"    for i in range(6)])

print(f"Feature vector size: {len(HC_COLS)}")


Feature vector size: 73


In [3]:
HC_CACHE = os.path.join(OUTPUT_DIR, "hc_features_cache.npz")

def collect_records(root):
    recs = []
    for split in ["train","val","test"]:
        sd = Path(root)/split
        if not sd.exists(): continue
        for sp in sorted(sd.iterdir()):
            if not sp.is_dir(): continue
            for p in sorted(sp.rglob("*")):
                if p.suffix.lower() in {".jpg",".jpeg",".png"}:
                    recs.append((str(p), sp.name, split))
    return recs

records     = collect_records(DATASET_ROOT)
all_paths   = [r[0] for r in records]
all_species = [r[1] for r in records]
all_splits  = np.array([r[2] for r in records])
print(f"Total images: {len(records)}")

# -- FIX: invalidate the cache if the dataset on disk no longer matches it --
# Previously this cache was never checked against the current file count, so
# after regenerating/fixing the dataset folder, this cell would silently keep
# loading stale features extracted from the OLD (corrupted) image set.
cache_valid = False
if os.path.exists(HC_CACHE):
    _cached = np.load(HC_CACHE, allow_pickle=True)
    if _cached["X"].shape[0] == len(records):
        cache_valid = True
    else:
        print(f"Cache has {_cached['X'].shape[0]} rows but dataset now has "
              f"{len(records)} images -- dataset changed, deleting stale cache.")
        os.remove(HC_CACHE)

if cache_valid:
    print("Cache matches current dataset size -- loading ...")
    X_hc = _cached["X"]
    print(f"Loaded: {X_hc.shape}")
else:
    from concurrent.futures import ProcessPoolExecutor, as_completed
    import multiprocessing
    def _ext(args):
        p, i = args
        try:    return i, extract_handcrafted(p)
        except: return i, np.zeros(len(HC_COLS))
    n_w = max(1, multiprocessing.cpu_count()-1)
    print(f"Extracting ({n_w} cores) ...")
    X_hc = np.zeros((len(all_paths), len(HC_COLS)), dtype=np.float32)
    with ProcessPoolExecutor(max_workers=n_w) as ex:
        futs = {ex.submit(_ext,(p,i)): i for i,p in enumerate(all_paths)}
        for fut in tqdm(as_completed(futs), total=len(all_paths)):
            idx, feat = fut.result()
            X_hc[idx] = feat
    np.savez_compressed(HC_CACHE, X=X_hc)
    print(f"Saved. Shape: {X_hc.shape}")


Total images: 19145
Cache matches current dataset size -- loading ...
Loaded: (19145, 73)


## Step 2 — Deep Feature Extraction (GPU)

In [4]:
import timm
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
print(f"timm {timm.__version__} -- transforms ready")

DEEP_CACHE = os.path.join(OUTPUT_DIR, "deep_features_cache.npz")
BATCH_SIZE = 64 if device.type == "cuda" else 32

class LeafDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths; self.transform = transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        try:
            return self.transform(Image.open(self.paths[idx]).convert("RGB"))
        except: return torch.zeros(3, 224, 224)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# -- FIX: invalidate the cache if the dataset on disk no longer matches it --
# Same stale-cache issue as the handcrafted-feature cache above: this used to
# load whatever was on disk regardless of whether the dataset had changed.
deep_cache_valid = False
if os.path.exists(DEEP_CACHE):
    _cached_deep = np.load(DEEP_CACHE, allow_pickle=True)
    if _cached_deep["X"].shape[0] == len(records):
        deep_cache_valid = True
    else:
        print(f"Deep-feature cache has {_cached_deep['X'].shape[0]} rows but "
              f"dataset now has {len(records)} images -- dataset changed, "
              f"deleting stale cache.")
        os.remove(DEEP_CACHE)

if deep_cache_valid:
    print("Cache matches current dataset size -- loading ...")
    X_deep = _cached_deep["X"]
    print(f"Loaded: {X_deep.shape}")
else:
    print(f"Extracting EfficientNetB3 on {device} ...")
    extractor = timm.create_model("efficientnet_b3",
                                   pretrained=True, num_classes=0).to(device)
    extractor.eval()
    loader  = DataLoader(LeafDataset(all_paths, transform),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=(device.type=="cuda"))
    X_deep  = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="EfficientNetB3"):
            feats = extractor(batch.to(device))
            X_deep.append(feats.cpu().numpy())
            if device.type == "cuda": torch.cuda.empty_cache()
    X_deep = np.concatenate(X_deep, axis=0).astype(np.float32)
    np.savez_compressed(DEEP_CACHE, X=X_deep)
    print(f"Saved. Shape: {X_deep.shape}")

X_all = np.concatenate([X_deep, X_hc], axis=1)
print(f"Fused: {X_all.shape}")


timm 1.0.27 -- transforms ready
Cache matches current dataset size -- loading ...
Loaded: (19145, 1536)
Fused: (19145, 1609)


## Step 3 — Save Train/Test Split

In [5]:
# -- Train/Val/Test Split -- PCA=300 -----------------------------------------
# This cell ALWAYS regenerates the split (deletes old cache first)
# Change N_PCA below to adjust PCA components

N_PCA = 300   # <- change this number to adjust PCA components (was 100)

SPLIT_CACHE = os.path.join(OUTPUT_DIR, "train_test_split.npz")

# Delete old cache so it regenerates with new PCA value
if os.path.exists(SPLIT_CACHE):
    os.remove(SPLIT_CACHE)
    print(f"Old cache deleted -- regenerating with PCA={N_PCA} ...")
else:
    print(f"No cache found -- generating with PCA={N_PCA} ...")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

try:
    from imblearn.over_sampling import SMOTE
    SMOTE_AVAILABLE = True
except ImportError:
    SMOTE_AVAILABLE = False
    print("SMOTE not available -- pip install imbalanced-learn")

# Collect records
def collect_records(root):
    recs = []
    for split in ["train","val","test"]:
        sd = Path(root)/split
        if not sd.exists(): continue
        for sp in sorted(sd.iterdir()):
            if not sp.is_dir(): continue
            for p in sorted(sp.rglob("*")):
                if p.suffix.lower() in {".jpg",".jpeg",".png"}:
                    recs.append((str(p), sp.name, split))
    return recs

records     = collect_records(DATASET_ROOT)
all_species = [r[1] for r in records]
all_splits  = np.array([r[2] for r in records])

le        = LabelEncoder()
y_species = le.fit_transform(all_species)

# -- FIX: keep val and test separate. Previously test_mask combined val+test,
# so "the held-out test split" reported in the Methods text was actually
# val+test merged. Val is now its own held-out set, distinct from test.
train_mask = all_splits == "train"
val_mask   = all_splits == "val"
test_mask  = all_splits == "test"

if train_mask.sum() > 10 and test_mask.sum() > 10:
    X_train_raw = X_all[train_mask]; y_train_raw = y_species[train_mask]
    X_val_raw   = X_all[val_mask];   y_val_raw   = y_species[val_mask]
    X_test_raw  = X_all[test_mask];  y_test_raw  = y_species[test_mask]
    print("Using original dataset split (train/val/test kept separate).")
else:
    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X_all, y_species, test_size=0.2, random_state=42, stratify=y_species)
    X_val_raw, y_val_raw = X_test_raw, y_test_raw  # no separate val in fallback
    print("Fallback: 80/20 stratified split (no separate val set).")

print(f"Train raw: {X_train_raw.shape}  "
      f"Val raw: {X_val_raw.shape}  Test raw: {X_test_raw.shape}")

# SMOTE
if SMOTE_AVAILABLE:
    k = max(1, min(5, int(np.bincount(y_train_raw).min())-1))
    X_train_sm, y_train_sm = SMOTE(random_state=42,
        k_neighbors=k).fit_resample(X_train_raw, y_train_raw)
    print(f"After SMOTE: {X_train_sm.shape[0]} training samples")
else:
    X_train_sm, y_train_sm = X_train_raw, y_train_raw

# PCA -- uses N_PCA defined above
n_components = min(N_PCA, X_train_sm.shape[1]-1)
sc1 = StandardScaler()
pca = PCA(n_components=n_components, random_state=42)
sc2 = StandardScaler()

X_train_sc = sc2.fit_transform(pca.fit_transform(sc1.fit_transform(X_train_sm)))
X_val_sc   = sc2.transform(pca.transform(sc1.transform(X_val_raw)))
X_test_sc  = sc2.transform(pca.transform(sc1.transform(X_test_raw)))

print(f"PCA: {X_train_sm.shape[1]} -> {n_components} components "
      f"({pca.explained_variance_ratio_.sum()*100:.1f}% variance)")
print(f"Train: {X_train_sc.shape}  Val: {X_val_sc.shape}  Test: {X_test_sc.shape}")

# Save
np.savez_compressed(SPLIT_CACHE,
    X_train_sc=X_train_sc, X_val_sc=X_val_sc, X_test_sc=X_test_sc,
    y_train_sm=y_train_sm, y_val_raw=y_val_raw, y_test_raw=y_test_raw,
    n_components=np.array(pca.n_components_))

for fname, obj in [("label_encoder_species.pkl", le),
                    ("sc1.pkl", sc1), ("sc2.pkl", sc2), ("pca.pkl", pca)]:
    with open(os.path.join(OUTPUT_DIR, fname), "wb") as f:
        pickle.dump(obj, f)

print(f"\nSplit saved with PCA={N_PCA} components.")
print(f"Classes: {len(le.classes_)} -- {list(le.classes_)}")


Old cache deleted -- regenerating with PCA=300 ...
Using original dataset split (train/val/test kept separate).
Train raw: (14407, 1609)  Val raw: (2372, 1609)  Test raw: (2366, 1609)
After SMOTE: 35000 training samples
PCA: 1609 -> 300 components (84.0% variance)
Train: (35000, 300)  Val: (2372, 300)  Test: (2366, 300)

Split saved with PCA=300 components.
Classes: 7 -- [np.str_('Lamium_pooled'), np.str_('Lavandula_angustifolia'), np.str_('Lavandula_pooled'), np.str_('Leucas_aspera'), np.str_('Mentha_pooled'), np.str_('Ocimum_pooled'), np.str_('Plectranthus_pooled')]


## Step 4 — Verify everything saved

In [6]:
files = ["hc_features_cache.npz","deep_features_cache.npz",
         "train_test_split.npz","label_encoder_species.pkl",
         "sc1.pkl","sc2.pkl","pca.pkl"]
print("Files in OUTPUT_DIR:")
for f in files:
    path = os.path.join(OUTPUT_DIR, f)
    exists = os.path.exists(path)
    size   = os.path.getsize(path)/1e6 if exists else 0
    print(f"  {'OK' if exists else 'MISSING':6s} {f}  ({size:.1f} MB)")
print("\nSetup complete. Now run notebooks 1-12 in any order.")


Files in OUTPUT_DIR:
  OK     hc_features_cache.npz  (4.6 MB)
  OK     deep_features_cache.npz  (108.6 MB)
  OK     train_test_split.npz  (44.3 MB)
  OK     label_encoder_species.pkl  (0.0 MB)
  OK     sc1.pkl  (0.0 MB)
  OK     sc2.pkl  (0.0 MB)
  OK     pca.pkl  (1.9 MB)

Setup complete. Now run notebooks 1-12 in any order.
